# E1 ST0_plan (local CPU): extraction (= steering), selection (fresh b0013 minus pool near-duplicates), evaluation (pool half)

In [ ]:
FORCE = False


In [ ]:
import glob, hashlib, json, os, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd

MYDRIVE = Path(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-*"))[0]) / "My Drive"
PT3 = MYDRIVE / "ez-penta" / "post_train_3"
FU = PT3 / "followups"
FU.mkdir(parents=True, exist_ok=True)


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []


def write_jsonl(path, rs):
    with open(path, "w", encoding="utf-8") as f:
        for r in rs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def folder_ok(d):
    d = Path(d)
    if not (d / "DONE").exists() or not (d / "manifest.json").exists():
        return False
    if (d / "DONE").read_text().strip() != sha256_file(d / "manifest.json"):
        return False
    man = json.loads((d / "manifest.json").read_text())
    return all((d / f).exists() and sha256_file(d / f) == m["sha256"] for f, m in man["files"].items())


In [ ]:
E1 = FU / "e1"
E1.mkdir(parents=True, exist_ok=True)
assert not ((E1 / "plan.DONE").exists() and not FORCE), "E1 plan.DONE exists"
st = PT3 / "steering"
assert (st / "plan.DONE").read_text().strip() == sha256_file(st / "plan.json")
splan = json.loads((st / "plan.json").read_text())
fplan = json.loads((FU / "plan.json").read_text())
assert (FU / "plan.DONE").read_text().strip() == sha256_file(FU / "plan.json")
EXTRACTION = splan["sets"]["extraction"]
bd = E1 / "selection_batch" / "b0013"
assert (bd / "DONE").read_text().strip() == sha256_file(bd / "manifest.json")
bman = json.loads((bd / "manifest.json").read_text())
for f, m in bman["files"].items():
    assert sha256_file(bd / f) == m["sha256"], f
ov = {o["seed_id"]: o for o in json.loads((bd / "pool_overlap.json").read_text())}
drop = sorted(s for s, o in ov.items() if o["max_cos_pool"] > 0.85)
SEL = [p for p in load_jsonl(bd / "prompts.jsonl") if p["seed_id"] not in set(drop)]
SELECTION = [{k: p[k] for k in ("prompt_id", "seed_id", "category", "severity", "kind", "culture", "framing", "lang", "prompt")} | {"split": "fresh_b0013"} for p in SEL]
print("selection:", len(SELECTION), "prompts;", len({p["seed_id"] for p in SELECTION}), "seeds; dropped near-duplicates of pool seeds:", len(drop))


In [ ]:
half = set(fplan["half_seeds"])
ok = set(fplan["paired_ok_half"])
evh = load_jsonl(FU / "inputs" / "eval_half.jsonl")
KEYS = ("prompt_id", "seed_id", "batch", "category", "subtopic", "form", "severity", "kind", "expected", "culture", "framing", "lang", "prompt")
EVAL_US = [{k: p[k] for k in KEYS} for p in evh if p["culture"] == "us"]
EVAL_JP = [{**{k: p[k] for k in KEYS}, "us_prompt_id": p["us_prompt_id"], "chunk": p["batch"]} for p in evh if p["culture"] == "japan" and p["seed_id"] in ok]
us_first_batch = {p["prompt_id"]: p["batch"] for p in EVAL_US}
ext_seeds, sel_seeds = {r["seed_id"] for r in EXTRACTION}, {r["seed_id"] for r in SELECTION}
eval_seeds = {r["seed_id"] for r in EVAL_US + EVAL_JP}
assert not (eval_seeds & ext_seeds) and not (eval_seeds & sel_seeds) and not (ext_seeds & sel_seeds)
assert all(r["seed_id"] in half for r in EVAL_US)
print("eval US", len(EVAL_US), "eval JP", len(EVAL_JP), "| disjointness OK")
plan = {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "hashes": {"steering_plan": sha256_file(st / "plan.json"), "fu_plan": sha256_file(FU / "plan.json"),
                   "selection_batch_manifest": sha256_file(bd / "manifest.json"), "pool_overlap": sha256_file(bd / "pool_overlap.json")},
        "counts": {"extraction": len(EXTRACTION), "selection": len(SELECTION), "eval_us": len(EVAL_US), "eval_jp": len(EVAL_JP)},
        "sets": {"extraction": EXTRACTION, "selection": SELECTION, "eval_us": EVAL_US, "eval_jp": EVAL_JP},
        "us_first_batch": us_first_batch, "selection_dropped_near_duplicates": drop}
(E1 / "plan.json").write_text(json.dumps(plan, ensure_ascii=False))
(E1 / "plan.DONE").write_text(sha256_file(E1 / "plan.json") + "\n")
print("E1 plan written", plan["counts"])
